In [ ]:
# Colab bootstrap: Daten automatisch laden
from pathlib import Path
import os, sys, zipfile, urllib.request, subprocess

GITHUB_USER = "Facuriy"
GITHUB_REPO = "agro-ai-kurs"
BRANCH = "main"
BLOCK_DIR = Path("01_fernerkundung_drohne_indices_DE")
ZIP_NAME = "01_fernerkundung_drohne_indices_DE.zip"
ZIP_URL = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}/raw/{BRANCH}/{ZIP_NAME}"

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    # Falls in Colab Pakete fehlen, kann die folgende Zeile aktiviert werden:
    # subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy pandas matplotlib"], check=False)
    if not BLOCK_DIR.exists():
        print("Lade Unterrichtsdaten:", ZIP_URL)
        urllib.request.urlretrieve(ZIP_URL, ZIP_NAME)
        with zipfile.ZipFile(ZIP_NAME, "r") as zf:
            zf.extractall(".")
    os.chdir(BLOCK_DIR)
else:
    if Path.cwd().name != BLOCK_DIR.name and BLOCK_DIR.exists():
        os.chdir(BLOCK_DIR)

print("Arbeitsordner:", Path.cwd())
print("Daten bereit:", [p.name for p in Path.cwd().iterdir() if p.is_dir()])

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("class_data_remote_sensing/remote_sensing_small.npz")
OUT = Path("class_outputs_remote_sensing")
OUT.mkdir(exist_ok=True)

data = np.load(DATA, allow_pickle=True)
cube = data["cube"].astype("float32")       # Reihenfolge: blue, green, red, rededge, nir
band_names = list(data["band_names"])
bands = {name: cube[i] for i, name in enumerate(band_names)}
print("Datenquelle:", str(data["source"]))
print("Cube-Form:", cube.shape, "= Bänder x Höhe x Breite")
print("Bänder:", band_names)

In [ ]:
def norm01(x, p1=2, p2=98):
    lo, hi = np.nanpercentile(x, [p1, p2])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

def rgb_stack(r, g, b):
    return np.dstack([norm01(r), norm01(g), norm01(b)])

true_rgb = rgb_stack(bands["red"], bands["green"], bands["blue"])
false_color = rgb_stack(bands["nir"], bands["red"], bands["green"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(true_rgb); ax[0].set_title("RGB: red/green/blue")
ax[1].imshow(false_color); ax[1].set_title("Falschfarbe: NIR/red/green")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def safe_index(a, b):
    return (a - b) / (a + b + 1e-6)

ndvi = safe_index(bands["nir"], bands["red"])
ndre = safe_index(bands["nir"], bands["rededge"])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
im0 = ax[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.1, vmax=0.9)
ax[0].set_title("NDVI")
plt.colorbar(im0, ax=ax[0], fraction=0.046)
im1 = ax[1].imshow(ndre, cmap="viridis", vmin=-0.1, vmax=0.6)
ax[1].set_title("NDRE")
plt.colorbar(im1, ax=ax[1], fraction=0.046)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

print("NDVI Mittelwert:", round(float(np.nanmean(ndvi)), 3))
print("NDRE Mittelwert:", round(float(np.nanmean(ndre)), 3))

In [ ]:
def block_mean(img, factor):
    h, w = img.shape
    h2, w2 = (h // factor) * factor, (w // factor) * factor
    x = img[:h2, :w2]
    return x.reshape(h2//factor, factor, w2//factor, factor).mean(axis=(1, 3))

def nearest_up(img, factor):
    return np.repeat(np.repeat(img, factor, axis=0), factor, axis=1)

ndvi_drone = ndvi
ndvi_sat10 = nearest_up(block_mean(ndvi, 4), 4)
ndvi_sat30 = nearest_up(block_mean(ndvi, 12), 12)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, img, title in zip(ax, [ndvi_drone, ndvi_sat10, ndvi_sat30],
                         ["Drohne: fein", "Satellit: mittel", "Landsat-artig: grob"]):
    im = a.imshow(img, cmap="RdYlGn", vmin=-0.1, vmax=0.9)
    a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# HIER ÄNDERN
NDVI_SCHWELLE = 0.50
NDRE_SCHWELLE = 0.22

vegetation = ndvi > 0.20
alarm = vegetation & ((ndvi < NDVI_SCHWELLE) | (ndre < NDRE_SCHWELLE))
alarm_pct = 100 * alarm.sum() / max(1, vegetation.sum())
print(f"Alarmfläche innerhalb Vegetation: {alarm_pct:.1f}%")

overlay = true_rgb.copy()
overlay[alarm] = [1.0, 0.05, 0.05]

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(true_rgb); ax[0].set_title("RGB")
ax[1].imshow(alarm, cmap="Reds"); ax[1].set_title("Alarmmaske")
ax[2].imshow(overlay); ax[2].set_title("Overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
rows = []
n_rows, n_cols = 3, 3
h, w = ndvi.shape
for i in range(n_rows):
    for j in range(n_cols):
        y0, y1 = int(i*h/n_rows), int((i+1)*h/n_rows)
        x0, x1 = int(j*w/n_cols), int((j+1)*w/n_cols)
        veg = vegetation[y0:y1, x0:x1]
        al = alarm[y0:y1, x0:x1]
        pct = 100 * (al & veg).sum() / max(1, veg.sum())
        rows.append({
            "Zone": f"{chr(65+i)}{j+1}",
            "Vegetationspixel": int(veg.sum()),
            "Alarmpixel": int((al & veg).sum()),
            "Alarm_%": round(float(pct), 1),
            "Empfehlung": "zuerst prüfen" if pct > 30 else ("beobachten" if pct > 15 else "niedrig"),
        })

report = pd.DataFrame(rows).sort_values("Alarm_%", ascending=False)
display(report)
report.to_csv(OUT / "zonen_report.csv", index=False)
print("Gespeichert:", OUT / "zonen_report.csv")